<a href="https://colab.research.google.com/github/deanbaranes/Cloud_Computing/blob/main/Tut7_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

print("OK – libraries loaded successfully.")


OK – libraries loaded successfully.


In [ ]:
# ==========================================
# Simple RAG demo – clean minimal version
# ==========================================

from typing import List, Dict
import numpy as np

# Try modern embedding model
try:
    from sentence_transformers import SentenceTransformer
    MODEL_AVAILABLE = True
except ImportError:
    MODEL_AVAILABLE = False
    from sklearn.feature_extraction.text import TfidfVectorizer


# ---------- Simple in-memory Vector Store ----------
class SimpleVectorStore:
    def __init__(self):
        self.texts: List[str] = []
        self.vectors: np.ndarray | None = None

    def add(self, texts: List[str], vectors: np.ndarray):
        self.texts.extend(texts)
        if self.vectors is None:
            self.vectors = vectors
        else:
            self.vectors = np.vstack([self.vectors, vectors])

    def search(self, query_vec: np.ndarray, top_k: int = 3) -> List[Dict]:
        if query_vec.ndim == 1:
            query_vec = query_vec.reshape(1, -1)

        if self.vectors is None or len(self.texts) == 0:
            return []

        # cosine similarity
        qn = query_vec / (np.linalg.norm(query_vec, axis=1, keepdims=True) + 1e-8)
        dn = self.vectors / (np.linalg.norm(self.vectors, axis=1, keepdims=True) + 1e-8)

        scores = (qn @ dn.T)[0]
        idx = np.argsort(scores)[::-1][:top_k]

        return [
            {"text": self.texts[i], "score": float(scores[i])}
            for i in idx
        ]


# ---------- Simple RAG Core ----------
class SimpleRAG:
    def __init__(self):
        self.store = SimpleVectorStore()

        if MODEL_AVAILABLE:
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.use_tfidf = False
        else:
            self.vectorizer = TfidfVectorizer()
            self.use_tfidf = True

        self.docs_added = False

    def _embed(self, texts: List[str]) -> np.ndarray:
        if self.use_tfidf:
            if not self.docs_added:
                vecs = self.vectorizer.fit_transform(texts).toarray()
            else:
                vecs = self.vectorizer.transform(texts).toarray()
        else:
            vecs = self.model.encode(texts)

        return np.array(vecs, dtype=np.float32)

    def add_documents(self, docs: List[str]):
        vectors = self._embed(docs)
        self.store.add(docs, vectors)
        self.docs_added = True

    def query(self, question: str, top_k: int = 3) -> Dict:
        if not self.docs_added:
            return {"answer": "No documents added yet.", "results": []}

        q_vec = self._embed([question])[0]
        results = self.store.search(q_vec, top_k=top_k)

        answer = "Top relevant texts:\n" + "\n".join(
            f"- {r['text']} (score={r['score']:.3f})"
            for r in results
        )

        return {"answer": answer, "results": results}


In [ ]:
# יצירת אובייקט RAG
rag = SimpleRAG()

# מסמכים לדוגמה (נוכל להחליף אחר כך במסמכים אמיתיים)
documents = [
    "Houseplants need the right balance of water and light to stay healthy.",
    "Succulents require very little water but need strong sunlight.",
    "Overwatering is a common cause of indoor plant problems."
]

# מוסיפים את המסמכים למנוע
rag.add_documents(documents)

# שואלים שאלה
question = "How should I water indoor plants?"
response = rag.query(question, top_k=2)

print("QUESTION:")
print(question)
print("\nANSWER:")
print(response["answer"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

QUESTION:
How should I water indoor plants?

ANSWER:
Top relevant texts:
- Overwatering is a common cause of indoor plant problems. (score=0.607)
- Houseplants need the right balance of water and light to stay healthy. (score=0.562)


In [ ]:
!pip install PyPDF2

import PyPDF2

def extract_text_from_pdf(path):
    text = ""
    with open(path, "rb") as f:
        pdf = PyPDF2.PdfReader(f)
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

# דוגמא — תבחר קובץ אחד
text1 = extract_text_from_pdf("/content/pdis-03-15-0340-fe.pdf")
text2 = extract_text_from_pdf("/content/s41598-024-52038-y.pdf")

print("Extracted characters:", len(text1))


In [ ]:
import os

os.listdir('/content')


['.config',
 '1-s2.0-S2772899424000417-main.pdf',
 's41598-025-98454-6.docx',
 's41598-025-98454-6.pdf',
 '1-s2.0-S2772899424000417-main(1).docx',
 'pdis-03-15-0340-fe.docx',
 's41598-025-98454-6(1).docx',
 'pdis-03-15-0340-fe(1).docx',
 'pdis-03-15-0340-fe.pdf',
 's41598-024-52038-y.docx',
 'fpls_07_01419_pdf.docx',
 '1-s2.0-S2772899424000417-main.docx',
 's41598-024-52038-y.pdf',
 'fpls_07_01419_pdf.pdf',
 'sample_data']

In [ ]:
import os

pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]
pdf_files


['1-s2.0-S2772899424000417-main.pdf',
 's41598-025-98454-6.pdf',
 'pdis-03-15-0340-fe.pdf',
 's41598-024-52038-y.pdf',
 'fpls_07_01419_pdf.pdf']

In [ ]:
!pip install PyPDF2

import PyPDF2
import os

def extract_text_from_pdf(path):
    text = ""
    with open(path, "rb") as f:
        pdf = PyPDF2.PdfReader(f)
        for page in pdf.pages:
            try:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
            except:
                pass
    return text

# --- רשימת PDFים ---
pdf_files = ['1-s2.0-S2772899424000417-main.pdf',
             's41598-025-98454-6.pdf',
             'pdis-03-15-0340-fe.pdf',
             's41598-024-52038-y.pdf',
             'fpls_07_01419_pdf.pdf']

# --- חילוץ טקסט ---
texts = {}

for filename in pdf_files:
    full_path = "/content/" + filename
    print("קורא:", filename)
    extracted = extract_text_from_pdf(full_path)
    texts[filename] = extracted
    print("אורך הטקסט:", len(extracted))
    print("---------------------------")

texts


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.1 MB/s eta 0:00:00
קורא: 1-s2.0-S2772899424000417-main.pdf
אורך הטקסט: 73704
---------------------------
קורא: s41598-025-98454-6.pdf
אורך הטקסט: 61329
---------------------------
קורא: pdis-03-15-0340-fe.pdf
אורך הטקסט: 69879
---------------------------
קורא: s41598-024-52038-y.pdf


אורך הטקסט: 43852
---------------------------
קורא: fpls_07_01419_pdf.pdf
אורך הטקסט: 45920
---------------------------


{'1-s2.0-S2772899424000417-main.pdf': "Research Article\nA real time monitoring system for accurate plant leaves disease detection\nusing deep learning\nKazi Naimur Rahman*, Sajal Chandra Banik , Raihan Islam , Arafath Al Fahim\nChittagong University of Engineering &Technology, CUET, 4349, Chittagong, Bangladesh\nARTICLE INFO\nKeywords:\nPlant disease detection\nConvolutional neural networkDeep learning modelComputer visionPerformance evaluationImage processingPlant pathogenABSTRACT\nAccurate and timely detection of plant diseases is crucial for sustainable agriculture and food security. This\nresearch presents a real-time monitoring system utilizing deep learning techniques to detect diseases in plant\nleaves with high accuracy. We combined several plant datasets, including the PlantVillage Dataset, resulting in a\ncomprehensive dataset of 30,945 images across eight plant types (potato, tomato, pepper bell, apple, corn, grape,peach, and rice) and 35 disease classes. Initially, a custo

In [ ]:
def chunk_text(text, max_len=500):
    chunks = []
    current = ""

    for sentence in text.split("."):
        sentence = sentence.strip()
        if not sentence:
            continue

        # אם המשפט מתאים לגודל — מצרפים
        if len(current) + len(sentence) < max_len:
            current += sentence + ". "
        else:
            # פותחים chunk חדש
            chunks.append(current.strip())
            current = sentence + ". "

    if current:
        chunks.append(current.strip())

    return chunks

# יצירת chunks לכל המאמרים שלך
all_chunks = []

for filename, text in texts.items():
    article_chunks = chunk_text(text)
    print(filename, "→", len(article_chunks), "chunks")
    all_chunks.extend(article_chunks)

print("\nסה״כ כמות Chunks:", len(all_chunks))


1-s2.0-S2772899424000417-main.pdf → 174 chunks
s41598-025-98454-6.pdf → 143 chunks
pdis-03-15-0340-fe.pdf → 159 chunks
s41598-024-52038-y.pdf → 100 chunks
fpls_07_01419_pdf.pdf → 108 chunks

סה״כ כמות Chunks: 684


In [ ]:
# יצירת מנוע RAG
rag = SimpleRAG()

# טעינת כל ה-chunks
rag.add_documents(all_chunks)

print("Loaded documents into RAG:", len(all_chunks))


Loaded documents into RAG: 684


In [ ]:
question = "What are the main methods used for detecting plant diseases?"
response = rag.query(question, top_k=5)

print(response["answer"])


Top relevant texts:
- Common methods for the diagnosis and detection ofplant diseases include visual plant disease estimation by human
raters, microscopic evaluation of morphology features to identify
pathogens, as well as molecular, serological, and microbiological
diagnostic techniques (Bock et al. 2010; Nutter 2001). Microscopic methods utilize pathogen morphology (spores, myce-
lium, and fruiting bodies) for disease diagnosis. Speciation keys and
identification schemes are available. (score=0.753)
- com/scientificreports/
Related work
In this section, the previous scientific articles tackled the DL models that widely used for solving the problem 
of the plant diseases detection. While, other works discussed how to carry the advanced technologies “IoT” and 
techniques on the traditional central pivot for the plant diseases treatment. (score=0.701)
- In 1936, Riker and Riker emphasizedthe difficulties in diagnosing and detecting plant diseases. They gavean overview of the strengths a

In [ ]:
rag.query("Explain deep learning methods used for plant disease detection.", top_k=5)


{'answer': 'Top relevant texts:\n- Deep Learning for Plant Diseases\nHistorically, disease identiﬁcation has been supported by\nagricultural extension organizations or other institutio ns, such\nas local plant clinics. In more recent times, such eﬀorts have\nadditionally been supported by providing information for\ndisease diagnosis online, leveraging the increasing Inter net\npenetration worldwide. (score=0.837)\n- Recent years have seen a rise in interest in applying deep learning models to detect plant  diseases11−14. Numer -\nous research has shown these models’ ability to increase the precision and effectiveness of disease  detection15−18. Only a few studies have been  published19−21, making developing smartphone  apps22\xa0specifically for plant disease \ndetection a relatively young field. (score=0.812)\n- A web and mobile application were developed based on the best-performing models,\nallowing users to insert or capture images of plant leaves, detect diseases, and receive trea

In [ ]:
# ==========================================
# CELL 2: IMPORT LIBRARIES
# ==========================================

import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Optional
import re
import time

# Check what packages are available
print("Checking available packages...")

# ChromaDB
try:
    import chromadb
    CHROMADB_AVAILABLE = True
    print("ChromaDB: Available")
except ImportError:
    CHROMADB_AVAILABLE = False
    print("ChromaDB: Not available (will use fallback)")

# SentenceTransformers
try:
    from sentence_transformers import SentenceTransformer
    TRANSFORMERS_AVAILABLE = True
    print("SentenceTransformers: Available")
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("SentenceTransformers: Not available (will use TF-IDF fallback)")


Checking available packages...
ChromaDB: Available
SentenceTransformers: Available


In [ ]:
# ==========================================
# CELL 2: IMPORT LIBRARIES
# ==========================================

import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Optional
import re
import time

# Check what packages are available
print("Checking available packages...")

# ChromaDB
try:
    import chromadb
    CHROMADB_AVAILABLE = True
    print("ChromaDB: Available")
except ImportError:
    CHROMADB_AVAILABLE = False
    print("ChromaDB: Not available (will use fallback)")

# SentenceTransformers
try:
    from sentence_transformers import SentenceTransformer
    TRANSFORMERS_AVAILABLE = True
    print("SentenceTransformers: Available")
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("SentenceTransformers: Not available (will use TF-IDF fallback)")


Checking available packages...
ChromaDB: Available
SentenceTransformers: Available


In [32]:
# ==========================================
# CELL 4: RAG Core System (Updated Terms)
# ==========================================

from sklearn.feature_extraction.text import TfidfVectorizer

class EcologicalRAG:
    """Main RAG system for scientific papers"""
    def __init__(self, openai_api_key=None):
        print("Initializing Ecological RAG System...")

        # ===============================
        # Embedding Model Setup
        # ===============================
        if TRANSFORMERS_AVAILABLE:
            try:
                self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
                self.use_transformers = True
                print("Loaded SentenceTransformer embeddings")
            except:
                self.use_transformers = False
                self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
                print("Using TF-IDF embeddings (fallback)")
        else:
            self.use_transformers = False
            self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
            print("Using TF-IDF embeddings")

        # ===============================
        # Vector Store Setup (ChromaDB or fallback)
        # ===============================
        if CHROMADB_AVAILABLE:
            try:
                client = chromadb.Client()
                try:
                    self.collection = client.get_collection("ecological_papers")
                    print("Loaded existing ChromaDB collection")
                except:
                    self.collection = client.create_collection("ecological_papers")
                    print("Created new ChromaDB collection")
                self.use_chromadb = True
            except:
                self.collection = SimpleVectorStore()
                self.use_chromadb = False
                print("Using simple vector store (fallback)")
        else:
            self.collection = SimpleVectorStore()
            self.use_chromadb = False
            print("Using simple vector store")

        # ===============================
        # Optional OpenAI Setup
        # ===============================
        self.use_openai = False
        if openai_api_key:
            try:
                import openai
                self.openai = openai
                self.openai.api_key = openai_api_key
                self.use_openai = True
                print("OpenAI configured")
            except Exception as e:
                print("Failed to configure OpenAI:", e)

        self.papers = []
        self.fitted = False

        print("RAG system ready!")

    # ===================================
    # Utility Functions
    # ===================================

    def preprocess_text(self, text):
        if not text:
            return ""
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"[^\w\s\-\.\(\)]", " ", text)
        return text.strip()

    # UPDATED TERMS — Deep Learning + Sensors + Agriculture
    def extract_entities(self, text):
        """Extract keywords (species, locations, methods)"""

        entities = {'species': [], 'locations': [], 'methods': []}

        # Species (Latin form)
        species = re.findall(r"\b[A-Z][a-z]+ [a-z]+\b", text)
        entities["species"] = list(set(species))[:3]

        # UPDATED Locations
        locations = re.findall(
            r"\b(field|greenhouse|leaf|indoor|outdoor|crop|laboratory|farm)\b",
            text,
            re.IGNORECASE
        )
        entities["locations"] = list(set(locations))[:3]

        # UPDATED Methods — Deep Learning, Sensors, IoT
        methods = re.findall(
            r"\b(CNN|Convolutional|Deep learning|SVM|sensor|IoT|spectral|thermal|model|detection)\b",
            text,
            re.IGNORECASE
        )
        entities["methods"] = list(set(methods))[:3]

        return entities

    def generate_embeddings(self, texts):
        if self.use_transformers:
            return self.embedding_model.encode(texts, show_progress_bar=False)
        else:
            if not self.fitted:
                self.tfidf.fit(texts)
                self.fitted = True
            return self.tfidf.transform(texts).toarray()

print("RAG core class defined (with updated terms)!")
print("Next: Run Cell 5")


RAG core class defined (with updated terms)!
Next: Run Cell 5


In [ ]:
test_rag = EcologicalRAG()


Initializing Ecological RAG System...
Loaded SentenceTransformer embeddings
Created new ChromaDB collection
RAG system ready!


In [ ]:
# ==========================================
# CELL 5: Data Loading Methods
# ==========================================

def add_load_papers_method():
    """Attach load_papers method to EcologicalRAG class"""

    def load_papers(self, papers_data):
        print(f"Loading {len(papers_data)} papers...")

        documents = []
        metadatas = []
        ids = []

        for i, paper in enumerate(papers_data):
            abstract = paper.get("abstract", "").strip()
            if not abstract:
                continue

            # Create combined text
            text = f"{paper.get('title', '')}. {abstract}"
            text = self.preprocess_text(text)

            # Extract small entities
            entities = self.extract_entities(text)

            metadata = {
                "title": paper.get("title", "Unknown"),
                "authors": paper.get("authors", "Unknown"),
                "journal": paper.get("journal", "Unknown"),
                "year": paper.get("year", 0),
                "species": ", ".join(entities["species"]),
                "locations": ", ".join(entities["locations"]),
                "methods": ", ".join(entities["methods"])
            }

            documents.append(text)
            metadatas.append(metadata)
            ids.append(f"paper_{i}")

        if not documents:
            print("No valid documents found!")
            return

        print("Generating embeddings...")
        embeddings = self.generate_embeddings(documents)

        print("Storing papers in vector database...")
        if self.use_chromadb:
            self.collection.add(
                embeddings=embeddings.tolist(),
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
        else:
            self.collection.add(
                embeddings=embeddings,
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )

        self.papers = papers_data
        print(f"Successfully loaded {len(documents)} papers!")

    # Attach function to class
    EcologicalRAG.load_papers = load_papers

print("Data loading method added!")
print("Next: Run Cell 6 (search + query methods)")


Data loading method added!
Next: Run Cell 6 (search + query methods)


In [ ]:
# ==========================================
# CELL 6: Search + Query Methods
# ==========================================

def add_search_methods():
    """Attach search + query methods to EcologicalRAG"""

    def search(self, query, n_results=3):
        """Search for relevant papers using embeddings"""
        query_clean = self.preprocess_text(query)
        query_vec = self.generate_embeddings([query_clean])

        # ChromaDB vs fallback store
        if self.use_chromadb:
            results = self.collection.query(
                query_embeddings=query_vec.tolist(),
                n_results=n_results
            )
        else:
            results = self.collection.query(
                query_embeddings=query_vec,
                n_results=n_results
            )

        return results

    def query(self, question, n_results=3):
        """Return a readable answer based on the retrieved papers"""

        results = self.search(question, n_results=n_results)

        docs = results["documents"][0]
        metas = results["metadatas"][0]
        ids = results["ids"][0]

        # Build clean readable output
        answer = "Top relevant papers:\n\n"
        for i in range(len(docs)):
            answer += f"📄 Paper {i+1} ({ids[i]}):\n"
            answer += f"Title: {metas[i].get('title','Unknown')}\n"
            answer += f"Authors: {metas[i].get('authors','Unknown')}\n"
            answer += f"Methods: {metas[i].get('methods','')}\n"
            answer += f"Snippet: {docs[i][:300]}...\n\n"

        return {
            "response": answer,
            "papers_found": len(docs),
            "raw": results
        }

    # attach methods
    EcologicalRAG.search = search
    EcologicalRAG.query = query

print("Search + Query methods added!")
print("Next: We load your own PDF papers (Cell 7–10)")


Search + Query methods added!
Next: We load your own PDF papers (Cell 7–10)


In [ ]:
# ==========================================
# CELL 7: PDF Loading Utility
# ==========================================

import PyPDF2

def load_pdf_as_text(path):
    """Extract all text from a PDF file"""
    text = ""
    try:
        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text += page.extract_text() + "\n"
    except Exception as e:
        print("Error reading PDF:", e)
    return text


In [ ]:
# ==========================================
# CELL 8: Convert PDFs into paper objects
# ==========================================

def pdfs_to_papers(pdf_file_paths):
    papers = []

    for i, pdf_path in enumerate(pdf_file_paths):
        print(f"Reading PDF: {pdf_path}")
        text = load_pdf_as_text(pdf_path)

        if len(text.strip()) < 50:
            print("⚠️ PDF seems empty or unreadable.")
            continue

        papers.append({
            "title": f"Paper {i+1}",
            "authors": "Unknown",
            "journal": "Unknown",
            "year": 2024,
            "abstract": text
        })

        print(f"✓ Loaded PDF #{i+1} ({len(text)} characters)\n")

    print(f"Total loaded papers: {len(papers)}")
    return papers


In [ ]:
# ==========================================
# CELL 9: List your PDFs
# ==========================================

pdf_files = [
    "1-s2.0-S2772899424000417-main.pdf",
    "s41598-025-98454-6.pdf",
    "pdis-03-15-0340-fe.pdf",
    "s41598-024-52038-y.pdf",
    "fpls_07_01419_pdf.pdf"
]

pdf_files


['1-s2.0-S2772899424000417-main.pdf',
 's41598-025-98454-6.pdf',
 'pdis-03-15-0340-fe.pdf',
 's41598-024-52038-y.pdf',
 'fpls_07_01419_pdf.pdf']

In [33]:
# ==========================================
# CELL 10: Build papers & Load into RAG
# ==========================================

# Convert PDFs to structured paper objects
papers_data = pdfs_to_papers(pdf_files)

# Attach load_papers method (from Cell 5)
add_load_papers_method()
add_search_methods()

# Initialize new RAG system
rag = EcologicalRAG()

# Load the papers into vector DB
rag.load_papers(papers_data)

print("\nRAG is ready with your PDF papers!")


Reading PDF: 1-s2.0-S2772899424000417-main.pdf
✓ Loaded PDF #1 (73704 characters)

Reading PDF: s41598-025-98454-6.pdf
✓ Loaded PDF #2 (61329 characters)

Reading PDF: pdis-03-15-0340-fe.pdf
✓ Loaded PDF #3 (69879 characters)

Reading PDF: s41598-024-52038-y.pdf


✓ Loaded PDF #4 (43852 characters)

Reading PDF: fpls_07_01419_pdf.pdf
✓ Loaded PDF #5 (45920 characters)

Total loaded papers: 5
Initializing Ecological RAG System...
Loaded SentenceTransformer embeddings
Loaded existing ChromaDB collection
RAG system ready!
Loading 5 papers...
Generating embeddings...
Storing papers in vector database...
Successfully loaded 5 papers!

RAG is ready with your PDF papers!


In [34]:
# ==========================================
# CELL 11: Gradio Web Interface (Corrected)
# ==========================================

# Check if Gradio is installed
try:
    import gradio as gr
    GRADIO_AVAILABLE = True
    print("Gradio: Available")
except ImportError:
    GRADIO_AVAILABLE = False
    print("Gradio: NOT available – please install it: pip install gradio")

# Only run if Gradio is available
if GRADIO_AVAILABLE:

    def gradio_query(question, n_results=3):
        """Query function for Gradio interface"""
        if not question.strip():
            return "Please enter a question."

        try:
            # 🔥 IMPORTANT FIX: use rag (your loaded PDF RAG system)
            result = rag.query(question, n_results=int(n_results))
            return result["response"]
        except Exception as e:
            return f"Error: {e}"


    def create_gradio_interface():
        """Create Gradio web UI"""
        examples = [
            ["What accuracy do CNN models achieve in plant classification?", 3],
            ["Which deep learning models are used for plant disease detection?", 3],
            ["What sensors are used for plant disease monitoring?", 3],
        ]

        interface = gr.Interface(
            fn=gradio_query,
            inputs=[
                gr.Textbox(label="Enter your question", placeholder="Ask about your uploaded scientific papers..."),
                gr.Slider(1, 10, value=3, step=1, label="Number of results")
            ],
            outputs=gr.Textbox(label="RAG Answer", lines=12),
            examples=examples,
            title="Plant Disease Detection RAG System",
            description="Ask any question based on your uploaded scientific papers."
        )

        interface.launch(share=True)


# Create the interface
create_gradio_interface()


Gradio: Available
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc75bbaff90b532ba6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
